In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
from tqdm.auto import tqdm

/home4/s6019595/.venv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset_df = pd.read_parquet("/home4/s6019595/nlp/data/dataset.parquet")
dataset_df

,problem,level,type,solution
0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",Level 5,Algebra,"For the piecewise function to be continuous, t..."
1,A rectangular band formation is a formation wi...,Level 5,Algebra,Let $x$ be the number of band members in each ...
2,What is the degree of the polynomial $(4 +5x^3...,Level 3,Algebra,This polynomial is not written in standard for...
3,Evaluate $\left\lceil3\left(6-\frac12\right)\r...,Level 3,Algebra,"Firstly, $3\left(6-\frac12\right)=18-1-\frac12..."
4,Sam is hired for a 20-day period. On days that...,Level 3,Algebra,Call $x$ the number of days Sam works and $y$ ...
...,...,...,...,...
12495,If $\sin x + \cos x = \frac{1}{5}$ and $0 < x ...,Level 5,Precalculus,"From the given equation, $\cos x = \frac{1}{5}..."
12496,The matrix for projecting onto a certain plane...,Level 5,Precalculus,Since $\begin{pmatrix} a \\ b \\ c \end{pmatri...
12497,"Let $\mathbf{a},$ $\mathbf{b},$ and $\mathbf{c...",Level 4,Precalculus,Since $\mathbf{a} + \mathbf{b} + \mathbf{c} = ...
12498,Find the smallest positive integer solution to...,Level 5,Precalculus,"By the tangent addition formula,\n\begin{align..."


In [3]:
dataset_df = pd.read_parquet("/home4/s6019595/nlp/data/dataset_aime.parquet")
dataset_df

,ID,Year,Problem Number,Question,Answer,Part
0,1983-1,1983,1,"Let $x$ , $y$ and $z$ all exceed $1$ and let $...",60,None
1,1983-2,1983,2,"Let $f(x)=|x-p|+|x-15|+|x-p-15|$ , where $0 < ...",15,None
2,1983-3,1983,3,What is the product of the real roots of the e...,20,None
3,1983-4,1983,4,A machine-shop cutting tool has the shape of a...,26,None
4,1983-5,1983,5,Suppose that the sum of the squares of two com...,4,None
...,...,...,...,...,...,...
928,2024-II-11,2024,11,Find the number of triples of nonnegative inte...,601,II
929,2024-II-12,2024,12,"Let $O(0,0),A(\tfrac{1}{2},0),$ and $B(0,\tfra...",23,II
930,2024-II-13,2024,13,Let $\omega\neq 1$ be a 13th root of unity. Fi...,321,II
931,2024-II-14,2024,14,Let $b \geq 2$ be an integer. Call a positive ...,211,II


In [4]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "../models/L1-Qwen-1.5B-Exact"
model_LCPO = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_path)


Loading checkpoint shards: 100%|██████████| 2/2 [00:10<00:00,  5.47s/it]


In [6]:
dataset_df.shape[0]

933

In [ ]:
states = []

subset = dataset_df.iloc[:660]

for index, q in tqdm(subset.iterrows(), total=len(subset), desc="Processing rows"):
    inputs = tokenizer(q["Question"], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_LCPO(**inputs, output_hidden_states=True)

    hidden_last = outputs.hidden_states[-1][:, -1, :]
    states.append(hidden_last.detach().to(torch.float16).cpu().numpy())

states = np.stack(states, axis=0)
np.save("hidden_states_aime.npy", states)


Processing rows: 100%|██████████| 933/933 [00:24<00:00, 38.79it/s]
